# Document Intelligence RAG System

A Google Colab portfolio project that answers questions grounded in a user-uploaded document.

**Stack:** PyTorch, Sentence Transformers, FAISS, TF-IDF, Hugging Face Transformers, Qwen 2.5 1.5B Instruct.

**Recommended runtime:** Google Colab with a T4 GPU. The first run downloads the models. No API key is required.

## Pipeline

1. Upload a PDF, DOCX, TXT, Markdown, CSV, or JSON file.
2. Extract its text and split it into overlapping chunks.
3. Generate embeddings and index them in FAISS.
4. Combine semantic search with TF-IDF keyword retrieval.
5. Use Qwen to answer from retrieved context.

> Before publishing to GitHub, clear notebook outputs and confirm no personal information, private documents, or secrets are included.

## 1. Install dependencies

Run this cell first. This project does not require an external API key.

In [ ]:
%pip install -q "transformers>=4.45.0" "sentence-transformers>=3.0.0" "faiss-cpu>=1.8.0" "scikit-learn>=1.4.0" "pypdf>=5.0.0" "python-docx>=1.1.0"

## 2. Import libraries and check the runtime

In [ ]:
import os
import re
from pathlib import Path
from datetime import datetime

import numpy as np
import torch
import faiss
from pypdf import PdfReader
from docx import Document
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline
from google.colab import files

USE_GPU = torch.cuda.is_available()
DEVICE = 0 if USE_GPU else -1

print("GPU available:", USE_GPU)
if USE_GPU:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. The model can run on CPU, but answers may be slow.")

## 3. Load the embedding model and local language model

The first run downloads both models. The language model uses the GPU when available.

In [ ]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda" if USE_GPU else "cpu"
)

model_dtype = torch.float16 if USE_GPU else torch.float32

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device=DEVICE,
    model_kwargs={"torch_dtype": model_dtype}
)

generator.tokenizer.padding_side = "left"
print("Embedding model and Qwen are ready.")

## 4. Define document extraction, chunking, and indexing

Supported formats: PDF, DOCX, TXT, MD, CSV, and JSON. Scanned PDFs without selectable text need OCR, which is not included in this baseline.

In [ ]:
SUPPORTED_EXTENSIONS = {".pdf", ".docx", ".txt", ".md", ".csv", ".json"}


def extract_document_text(file_path):
    file_path = str(file_path)
    extension = Path(file_path).suffix.lower()

    if extension not in SUPPORTED_EXTENSIONS:
        raise ValueError(
            f"Unsupported file type: {extension}. Supported types: {', '.join(sorted(SUPPORTED_EXTENSIONS))}"
        )

    if extension == ".pdf":
        reader = PdfReader(file_path)
        parts = []
        for page_number, page in enumerate(reader.pages, start=1):
            page_text = page.extract_text() or ""
            if page_text.strip():
                parts.append(f"[Page {page_number}]\n{page_text.strip()}")
        text = "\n\n".join(parts)

    elif extension == ".docx":
        doc = Document(file_path)
        parts = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
        for table in doc.tables:
            for row in table.rows:
                row_text = " | ".join(cell.text.strip() for cell in row.cells)
                if row_text.strip(" |"):
                    parts.append(row_text)
        text = "\n".join(parts)

    else:
        with open(file_path, "r", encoding="utf-8", errors="replace") as file:
            text = file.read()

    if not text.strip():
        raise ValueError("No readable text was extracted. The PDF may be scanned or the file may be empty.")
    return text


def create_chunks(text, chunk_words=160, overlap_words=35):
    words = text.split()
    if not words:
        return []
    step = chunk_words - overlap_words
    chunks = []
    for start in range(0, len(words), step):
        chunk = " ".join(words[start:start + chunk_words]).strip()
        if chunk:
            chunks.append(chunk)
        if start + chunk_words >= len(words):
            break
    return chunks


def index_document_text(text, document_name="uploaded document"):
    global pdf_text, pdf_chunks, pdf_embeddings, pdf_index
    global tfidf_vectorizer, tfidf_matrix, current_document

    new_chunks = create_chunks(text)
    if not new_chunks:
        raise ValueError("No chunks were created from this document.")

    print(f"Creating embeddings for {len(new_chunks)} chunks...")
    new_embeddings = embedding_model.encode(
        new_chunks,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
        batch_size=32
    ).astype(np.float32)

    new_index = faiss.IndexFlatIP(new_embeddings.shape[1])
    new_index.add(new_embeddings)

    new_vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        sublinear_tf=True
    )
    new_tfidf_matrix = new_vectorizer.fit_transform(new_chunks)

    pdf_text = text
    pdf_chunks = new_chunks
    pdf_embeddings = new_embeddings
    pdf_index = new_index
    tfidf_vectorizer = new_vectorizer
    tfidf_matrix = new_tfidf_matrix
    current_document = document_name

    print("\nDocument indexed successfully!")
    print("File:", current_document)
    print("Characters:", len(pdf_text))
    print("Chunks:", len(pdf_chunks))
    print("FAISS vectors:", pdf_index.ntotal)


def upload_new_document():
    print("Choose one PDF, DOCX, TXT, MD, CSV, or JSON file.")
    uploaded = files.upload()
    if not uploaded:
        print("No file was uploaded.")
        return

    supported = [
        name for name in uploaded
        if Path(name).suffix.lower() in SUPPORTED_EXTENSIONS
    ]
    if not supported:
        raise ValueError(f"Please upload one of: {', '.join(sorted(SUPPORTED_EXTENSIONS))}")

    file_name = supported[0]
    new_text = extract_document_text(file_name)
    index_document_text(new_text, document_name=file_name)

    if len(supported) > 1:
        print("Only the first supported file was indexed. Run this function again for another file.")

## 5. Upload your first document

Run this cell and choose a document. To switch to another document later, call `upload_new_document()` again; this rebuilds the index while keeping the models loaded.

In [ ]:
upload_new_document()

## 6. Optional: extract and sort dates from timeline-style documents

This helper is separate from general question answering. It works best when dates and milestones appear on their own lines or beside their descriptions.

In [ ]:
MONTHS = (
    r"Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|"
    r"Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?"
)

EXACT_DATE_LINE = re.compile(
    rf"^\s*(\d{{1,2}}\s+(?:{MONTHS})\s+\d{{4}})(?:\s*[—–:-]\s*(.*))?\s*$",
    re.IGNORECASE
)
MONTH_PERIOD_LINE = re.compile(
    rf"^\s*((?:{MONTHS})(?:\s*[-–]\s*(?:{MONTHS}))?\s+\d{{4}})(?:\s*[—–:-]\s*(.*))?\s*$",
    re.IGNORECASE
)


def extract_document_dates(text):
    lines = [re.sub(r"\s+", " ", line).strip() for line in text.splitlines() if line.strip()]
    found = []
    status_words = {"completed", "planned", "in progress", "done"}

    def is_date_or_page(line):
        return bool(
            EXACT_DATE_LINE.fullmatch(line)
            or MONTH_PERIOD_LINE.fullmatch(line)
            or re.match(r"^\[Page\s+\d+\]$", line, re.IGNORECASE)
        )

    for i, line in enumerate(lines):
        exact_match = EXACT_DATE_LINE.fullmatch(line)
        period_match = MONTH_PERIOD_LINE.fullmatch(line)
        date_text, event = None, ""

        if exact_match:
            date_text = exact_match.group(1).strip()
            event = (exact_match.group(2) or "").strip()
            if not event:
                for candidate in lines[i + 1:i + 4]:
                    if candidate.lower() in status_words:
                        continue
                    if is_date_or_page(candidate):
                        break
                    event = candidate
                    break

        elif period_match:
            date_text = period_match.group(1).strip()
            event = (period_match.group(2) or "").strip()
            if not event:
                for candidate in reversed(lines[max(0, i - 3):i]):
                    if is_date_or_page(candidate):
                        break
                    if candidate.lower() in {"roadmap", "timeline", "target date", "date", "milestone"}:
                        continue
                    event = candidate
                    break
                if not event:
                    for candidate in lines[i + 1:i + 3]:
                        if is_date_or_page(candidate):
                            break
                        event = candidate
                        break

        if date_text and event:
            found.append((date_text, event))

    grouped = {}
    for date_text, event in found:
        key = re.sub(r"\s+", " ", date_text).lower()
        if key not in grouped:
            grouped[key] = [date_text, []]
        if event not in grouped[key][1]:
            grouped[key][1].append(event)

    def sort_key(item):
        date_text = item[0]
        match = re.fullmatch(r"(\d{1,2})\s+([A-Za-z]+)\s+(\d{4})", date_text)
        if match:
            day, month, year = match.groups()
            try:
                dt = datetime.strptime(f"{day} {month[:3]} {year}", "%d %b %Y")
                return (dt.year, dt.month, dt.day)
            except ValueError:
                pass

        match = re.fullmatch(r"([A-Za-z]+)(?:\s*[-–]\s*([A-Za-z]+))?\s+(\d{4})", date_text)
        if match:
            month, _, year = match.groups()
            try:
                dt = datetime.strptime(f"{month[:3]} {year}", "%b %Y")
                return (dt.year, dt.month, 1)
            except ValueError:
                pass
        return (9999, 12, 31)

    ordered = sorted(grouped.values(), key=sort_key)
    return [f"{date_text} — {'; '.join(events)}" for date_text, events in ordered]


def show_document_timeline():
    if "pdf_text" not in globals() or not pdf_text.strip():
        print("Upload and index a document first.")
        return
    entries = extract_document_dates(pdf_text)
    if entries:
        print("\n".join(entries))
    else:
        print("No standalone date/milestone lines were detected. Try asking with ask_pdf().")

## 7. Ask questions about the current document

`ask_pdf()` uses hybrid retrieval for longer documents and the full text for shorter documents. A local model can still make mistakes; verify important answers against the original document.

In [ ]:
def ask_pdf(question, top_k=8, max_context_chars=6500):
    if not isinstance(question, str) or not question.strip():
        print("Please enter a question.")
        return

    if "pdf_text" not in globals() or not pdf_text.strip():
        print("Upload a document first by running upload_new_document().")
        return

    q = question.lower().strip()
    timeline_terms = (
        "timeline", "milestone", "milestones", "roadmap",
        "planned future", "pilot dates", "rollout phases",
        "project dates", "project milestone dates",
        "planned pilot", "rollout timeline"
    )

    # Explicit timeline questions use direct extraction to reduce invented dates.
    # A question like "What is the CEO's date of birth?" still uses general RAG.
    if any(term in q for term in timeline_terms):
        entries = extract_document_dates(pdf_text)
        if entries:
            print("\n".join(entries))
            return

    # Use the whole text for small documents; retrieve relevant chunks for larger files.
    if len(pdf_text) <= 12000:
        context = pdf_text
    else:
        question_vector = embedding_model.encode(
            [question],
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype(np.float32)[0]

        semantic_scores = pdf_embeddings @ question_vector
        score_range = float(np.ptp(semantic_scores))
        if score_range > 1e-8:
            semantic_scores = (semantic_scores - semantic_scores.min()) / score_range
        else:
            semantic_scores = np.zeros_like(semantic_scores)

        keyword_scores = cosine_similarity(
            tfidf_vectorizer.transform([question]),
            tfidf_matrix
        ).ravel()

        combined_scores = 0.60 * semantic_scores + 0.40 * keyword_scores
        ranked_ids = np.argsort(combined_scores)[::-1]
        selected_ids = set()
        for idx in ranked_ids[:top_k]:
            idx = int(idx)
            for neighbor in (idx - 1, idx, idx + 1):
                if 0 <= neighbor < len(pdf_chunks):
                    selected_ids.add(neighbor)

        context_parts = []
        total_chars = 0
        for idx in sorted(selected_ids):
            chunk = pdf_chunks[idx].strip()
            if not chunk or total_chars + len(chunk) > max_context_chars:
                continue
            context_parts.append(chunk)
            total_chars += len(chunk)
        context = "\n\n".join(context_parts)

    if not context.strip():
        print("No relevant document context was found.")
        return

    messages = [
        {
            "role": "system",
            "content": (
                "Answer using only the supplied document context. Give a direct, accurate, complete answer. "
                "For list or count questions, include relevant items supported by the context. Preserve names, "
                "dates, numbers and units. Do not infer missing facts or invent exact dates. If the answer is "
                "absent, say 'Not specified in the document.' Return only the answer, without source labels."
            )
        },
        {
            "role": "user",
            "content": f"DOCUMENT CONTEXT:\n{context}\n\nQUESTION:\n{question}"
        }
    ]

    prompt = generator.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    result = generator(
        prompt,
        max_new_tokens=220,
        do_sample=False,
        return_full_text=False,
        pad_token_id=generator.tokenizer.eos_token_id
    )

    answer = result[0]["generated_text"].strip()
    print(answer if answer else "No answer was generated.")

print("Question-answering function ready.")

## 8. Try example questions

Run one question at a time in a new cell. These examples fit the included fictional sample report.

```python
ask_pdf("Summarize this document.")
ask_pdf("How many accounts were in the evaluation dataset, and how many cancellations were recorded?")
ask_pdf("Which model was selected, and what were its ROC AUC and precision?")
ask_pdf("What is the approved churn-risk threshold and prediction window?")
ask_pdf("What are the project names and how many projects are mentioned?")
ask_pdf("What is the CEO's date of birth?")
ask_pdf("List all project milestone dates and planned future dates.")
```

For a separate timeline listing, run `show_document_timeline()`.

In [ ]:
# Example test — edit this question as needed.
ask_pdf("Summarize this document.")

## 9. Switch to another document

Run this command when you want to upload a different file:

```python
upload_new_document()
```

It extracts the new file and rebuilds the chunks, embeddings, FAISS index, and TF-IDF index while keeping the loaded models. Then use the same `ask_pdf()` function.

## Before publishing to GitHub

1. Use **Edit → Clear all outputs** in Colab so personal information and test answers are not saved in notebook outputs.
2. Save the notebook as `Document_Intelligence_RAG.ipynb`.
3. Never commit personal résumés, private documents, or secrets.
4. Include only synthetic or authorized sample files in `sample_data/`.
5. Note that the first run downloads models and a GPU is recommended.

## Limitations

- Scanned PDFs require OCR, which is not implemented in this baseline.
- Complex tables and page layouts may not extract perfectly.
- Small local models can return incomplete or inaccurate answers, especially for broad questions. Verify important answers against the document.
- Colab runtime resets clear the models and document index; rerun the notebook from the top to restore them.